# Exercise 0 — What is a Tensor?
A tensor is a multi-dimensional array that PyTorch uses to store data and perform mathematical operations on the GPU.

---

# Exercise 1 — Why do we need all five steps?
Prediction gives an output, loss measures error, gradients show how to change parameters, the optimizer updates them, and gradient reset prevents updates from stacking incorrectly.

### Solution — What would happen if we made predictions but never calculated a loss?

Without a loss function, the model receives no feedback about how wrong its predictions are.  
It can produce outputs, but it has no signal telling it how to improve.  
Training would not progress, and the model would never learn.

### Solution — What if we calculated gradients but never updated the parameters?

Gradients describe how the parameters should change, but if we never apply them, the model stays exactly the same.  
The loss will not decrease, and the model will not improve over time.  
Training becomes a static loop with no learning.

### Solution — What if we updated parameters but never reset the gradients?

If gradients are not reset, they accumulate across iterations.  
This causes updates to become larger and larger, often leading to unstable training or divergence.  
Resetting gradients ensures each update is based only on the current batch.

---

# Exercise 2 — Batches and Epochs

A **batch** is a small subset of the dataset that the model processes in one training iteration.  
An **epoch** means that the entire dataset has passed through the neural network once. Every sample (batch) has been used to make predictions, compute loss, calculate gradients, and update the model’s parameters. Because a single pass is rarely enough for the model to learn meaningful patterns, training typically involves many epochs.

Training on the entire dataset at once would be slow and require too much memory. Using batches makes training more efficient by reducing memory usage and allowing the model to update its parameters more frequently. The small amount of noise introduced by batch-based gradients often improves generalization, helping the model learn better overall.

 ---
# Exercise 3 - Why is the default datatype for tensors float32?

Neural networks use float32 because the entire learning process is based on continuous values.
The loss function is continuous (even for binary labels), and the gradients computed from this loss are also continuous.
During training, the weights are updated in small decimal steps, which requires a floating‑point datatype capable of representing these small changes.

Integers cannot store decimals and would therefore prevent both gradient calculations and weight updates.
For this reason, training data and model parameters must use a floating‑point format such as float32.

 ---
# Exercise 4 – Loss, Gradients and Backpropagation

The **loss** is a single value that measures how wrong the model’s predictions are.  
The **gradient** is the derivative of the loss with respect to each parameter and shows how the parameters should change to reduce the loss.  
**Backpropagation** is the algorithm that computes these gradients by applying the chain rule backward through the network.  
The **optimizer** uses the gradients to update the parameters, completing the learning loop.

 ---
# Exercise 5 – Dot-product influence on neural networks

The dot-product requires that the number of columns in the input tensor matches the number of rows in the weight tensor. Because of this, each layer transforms the input into a new tensor with a new shape determined by the weight matrix. This is how neural networks change the “resolution” or dimensionality of the data as it moves deeper through the layers. In other words, the dot-product rule controls how information is transferred and reshaped from one layer to the next.

 ---
# Exercise 6 – The Argmax function

When you use dim = 0 in a reduction operation such as argmax(), PyTorch collapses the tensor vertically. This means it compares the values column by column instead of row by row. For each column, PyTorch looks at all rows and returns the index of the row that contains the highest value in that column. The output is therefore a vector where each element represents the position of the maximum value within its respective column.

 ---
# Exercise 8 – The @ operator

The  @ operator  performs matrix multiplication between two tensors. It takes the rows of the first tensor and multiplies them with the columns of the second tensor to produce a new tensor containing the combined weighted values.

 ---
# Exercise 9 - Tensor Shapes and Broadcasting Logic

- X @ W has shape (100, 1) because (100,1) @ (1,1) → (100,1).
- Adding b works because b has shape (1,), which broadcasts to (100,1).
- If W had shape (1,), matrix multiplication would fail since @ expects 2D shapes.

A shape mismatch happens when inner dimensions don’t align, e.g. (100,2) @ (1,1).
Fix: make W shape (2,1) so multiplication is valid.

Understanding shapes is essential because every layer in a neural network
depends on correct dimensional alignment. Wrong shapes = wrong math.

 ---
# Exercise 10 - Reasoning About Autograd and the Computation Graph

The graph consists of:
X → matmul → z → add b → y_hat → subtract y_true → square → mean → loss.

W and b require gradients; X and y_true typically do not.

PyTorch computes gradients by chaining local derivatives backward from loss
through each operation.

If W.requires_grad=False → no gradient for W.
If backward() is missing → no gradients at all.
If zero_grad() is missing → gradients accumulate and distort updates.

 ---
# Exercise 11 - Understandning the Meaning of the Gradient

A large positive W.grad means increasing W increases the loss.
Gradient descent will therefore decrease W.

A very small gradient means the loss surface is flat → slow learning.
A very large gradient means steep surface → unstable updates.

Small gradients may require a higher learning rate.
Large gradients may require a lower learning rate or clipping.

 ---
# Exercise 12 - Reasoning About the Training Loop

The order matters:

1. Forward pass → you need predictions.
2. Loss → you need an error signal.
3. Backward → you need gradients before updating.
4. Update → uses the gradients.
5. Zero_grad → prepares for the next iteration.

Updating must be inside torch.no_grad() so the update itself is not tracked
as part of the computation graph.

Gradients accumulate by default so multiple backward passes can be combined,
but in a simple loop they must be cleared each iteration.

 ---
# Exercise 13 - Convergence, Stability and Learning Dynamics

Convergence means the loss stops improving and parameter updates become small.

The model may not reach exact true parameters because:
- Data contains noise
- Learning rate is not ideal
- Numerical precision or model limitations

Larger learning rate → faster but riskier.
Smaller learning rate → slower but stable.
Noisy data → imperfect convergence.
Bad initialization → slower or unstable learning.

Underfitting: high loss everywhere.
Overfitting: low training loss, high validation loss.
Correct learning: both losses decrease and stabilize together.

 ---
# Exercise 14 - Final exercise, UNet

In [ ]:
# We import PyTorch, which is the deep learning framework we use.
import torch

# We import the neural network module (nn) which contains layers, activations, etc.
import torch.nn as nn

# We import the optim module which contains optimizers like Adam, SGD, etc.
import torch.optim as optim


# ---------------------------------------------------------
# Building block: Conv → ReLU → Conv → ReLU
# This is the fundamental block used throughout the UNet.
# ---------------------------------------------------------
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        # Initialize the parent class nn.Module
        super().__init__()

        # We define a sequence of layers that will run in order.
        # nn.Sequential lets us group layers into a single "block".
        self.block = nn.Sequential(
            # First convolution layer:
            # - Takes 'in_channels' input channels
            # - Produces 'out_channels' feature maps
            # - Uses kernel size 3 (common in CNNs)
            # - padding=1 keeps the spatial size the same
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),

            # ReLU activation introduces non-linearity
            nn.ReLU(inplace=True),

            # Second convolution layer:
            # - Takes the output of the previous conv
            # - Produces the same number of channels (out_channels)
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),

            # Another ReLU activation
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        # The forward method defines how data flows through the block.
        # We simply pass x through the sequential block.
        return self.block(x)


# ---------------------------------------------------------
# Minimal UNet
# This is a simplified version of the UNet architecture.
# ---------------------------------------------------------
class UNet(nn.Module):
    def __init__(self):
        # Initialize the parent class
        super().__init__()

        # -------------------------
        # Encoder (Downsampling path)
        # -------------------------

        # First encoder block:
        # Input: 1 channel (e.g., grayscale image)
        # Output: 64 feature maps
        self.down1 = DoubleConv(1, 64)

        # MaxPool reduces spatial size by factor 2 (downsampling)
        self.pool1 = nn.MaxPool2d(2)

        # Second encoder block:
        # Input: 64 channels
        # Output: 128 channels
        self.down2 = DoubleConv(64, 128)

        # Another downsampling
        self.pool2 = nn.MaxPool2d(2)

        # -------------------------
        # Bottleneck (Deepest part)
        # -------------------------

        # This is the middle of the UNet where the representation is most compressed.
        self.bottleneck = DoubleConv(128, 256)

        # -------------------------
        # Decoder (Upsampling path)
        # -------------------------

        # First upsampling:
        # ConvTranspose2d doubles the spatial size (reverse of pooling)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)

        # After upsampling, we concatenate skip connections,
        # so input channels become 128 (upsampled) + 128 (skip) = 256
        self.conv2 = DoubleConv(256, 128)

        # Second upsampling:
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)

        # Again, concatenate skip connection:
        # 64 (upsampled) + 64 (skip) = 128
        self.conv1 = DoubleConv(128, 64)

        # -------------------------
        # Output layer
        # -------------------------

        # Final 1x1 convolution:
        # Reduces channels from 64 → 1 (e.g., segmentation mask)
        self.out = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        # -------------------------
        # Encoder
        # -------------------------

        # First down block
        x1 = self.down1(x)   # Save for skip connection
        x2 = self.pool1(x1)  # Downsample

        # Second down block
        x3 = self.down2(x2)  # Save for skip connection
        x4 = self.pool2(x3)  # Downsample

        # -------------------------
        # Bottleneck
        # -------------------------
        x5 = self.bottleneck(x4)

        # -------------------------
        # Decoder
        # -------------------------

        # First upsampling
        x6 = self.up2(x5)

        # Concatenate skip connection from encoder
        # dim=1 means we concatenate along the channel dimension
        x6 = torch.cat([x6, x3], dim=1)

        # Apply double conv block
        x6 = self.conv2(x6)

        # Second upsampling
        x7 = self.up1(x6)

        # Concatenate skip connection
        x7 = torch.cat([x7, x1], dim=1)

        # Apply double conv block
        x7 = self.conv1(x7)

        # Final output layer
        return self.out(x7)


# ---------------------------------------------------------
# Instantiate model, loss, optimizer
# ---------------------------------------------------------

# Create an instance of the UNet model
model = UNet()

# Mean Squared Error loss:
# Measures how close predictions are to target values
loss_fn = nn.MSELoss()

# Adam optimizer:
# - model.parameters() gives all trainable weights
# - lr=0.001 is the global learning rate
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ---------------------------------------------------------
# Dummy example data
# ---------------------------------------------------------

# Create a random input tensor:
# Shape: (batch_size=1, channels=1, height=128, width=128)
x = torch.randn(1, 1, 128, 128)

# Create a random target tensor of the same shape
y_true = torch.randn(1, 1, 128, 128)

# ---------------------------------------------------------
# Training loop
# ---------------------------------------------------------

epochs = 20  # Number of training iterations

for epoch in range(epochs):

    # -------------------------
    # 1. Forward pass
    # -------------------------
    # Pass input through the model to get predictions
    y_hat = model(x)

    # -------------------------
    # 2. Compute loss
    # -------------------------
    # Compare predictions with ground truth
    loss = loss_fn(y_hat, y_true)

    # -------------------------
    # 3. Backpropagation engine
    # -------------------------

    # Clear old gradients (PyTorch accumulates by default)
    optimizer.zero_grad()

    # Compute gradients of loss w.r.t. all model parameters
    loss.backward()

    # Update model parameters using the optimizer
    optimizer.step()

    # -------------------------
    # 4. Print progress
    # -------------------------
    if epoch % 5 == 0:
        # loss.item() extracts the Python float from the tensor
        print(f"Epoch {epoch:02d}: Loss = {loss.item():.4f}")


## DoubleConv Block (Conv → ReLU → Conv → ReLU)

This block is the fundamental building unit of the UNet.  
It applies two convolutional layers back‑to‑back, each followed by a ReLU activation.

- The convolutions extract spatial features from the image  
- Padding keeps the height and width unchanged  
- ReLU introduces non‑linearity so the model can learn complex patterns  
- Using two convolutions in a row allows the network to refine features at each resolution level

This block is used everywhere in the encoder, bottleneck, and decoder.

 ---
## Encoder Block (Downsampling Path)

Each encoder step consists of:

1. A DoubleConv block  
   - Extracts features at the current resolution  
   - Learns increasingly abstract representations

2. A MaxPool2d(2) layer  
   - Reduces spatial size by a factor of 2  
   - Keeps the most important activations  
   - Allows the network to see a larger context

The encoder compresses the image while increasing the number of feature channels.

 ---
## Bottleneck (The Deepest Part of the UNet)

The bottleneck is a DoubleConv block with the highest number of channels.

- It operates at the smallest spatial resolution  
- It captures global context  
- It acts as the bridge between encoder and decoder  

This is where the network has the most abstract understanding of the input.

 ---
## Decoder Block (Upsampling Path)

Each decoder step consists of:

1. A ConvTranspose2d layer  
   - Doubles the spatial resolution  
   - Reverses the effect of max pooling  
   - Often called “deconvolution” or “upsampling convolution”

2. A concatenation with the corresponding encoder feature map  
   - These are the skip connections  
   - They restore fine‑grained spatial details lost during downsampling  
   - They help the model produce sharp, accurate outputs

3. A DoubleConv block  
   - Refines the combined features  
   - Learns how to merge high‑level and low‑level information

The decoder reconstructs the image step by step.

 ---
## Skip Connections (The Key Idea of UNet)

Skip connections link encoder layers to decoder layers at the same resolution.

They allow the model to:

- Recover spatial details lost during pooling  
- Combine low‑level textures with high‑level semantics  
- Train deeper networks without vanishing gradients  

This is what gives UNet its characteristic “U” shape.

 ---
## Output Layer (1×1 Convolution)

The final layer is a Conv2d with:

- Kernel size 1  
- Output channels = 1 (for a single‑channel prediction)

This layer:

- Converts the 64‑channel feature map into the final output  
- Does not change spatial resolution  
- Acts as a linear projection from features → prediction

Common outputs:
- Segmentation mask  
- Denoised image  
- Reconstructed image  

 ---
## Training Loop (Forward → Loss → Backward → Update)

The training loop follows the standard PyTorch pattern:

1. **Forward pass**  
   - Input goes through the model  
   - Produces predictions (y_hat)

2. **Compute loss**  
   - Compares predictions to ground truth  
   - MSELoss measures pixel‑wise error

3. **Backward pass**  
   - optimizer.zero_grad() clears old gradients  
   - loss.backward() computes new gradients  
   - optimizer.step() updates all weights

4. **Progress print**  
   - Shows how the loss changes over time

This loop is identical for all neural networks in PyTorch.

